# 🧪 TravelMate AI - 25 Final System Testing

This notebook performs the final **end-to-end validation** before deployment.

We test the complete architecture:

```text
User request
    ↓
FastAPI
    ↓
Personalized Hybrid Recommender
    ↓
Itinerary Generator
    ↓
Validation
    ↓
Final API response
```

## Test categories

```text
1. 🏙️ All supported cities
2. 🎯 Multiple traveller profiles
3. 🤖 Recommendation response structure
4. 📅 Itinerary response structure
5. ⏱️ Daily time constraints
6. 🗺️ Coordinate availability
7. 🔁 API repeatability
8. 🚫 No Manali-only restriction
9. ❌ Invalid city handling
10. 🌐 HTTP-level endpoint testing
```

The goal is not to prove real-world recommendation accuracy. It is to verify that the **application architecture works correctly and consistently**.


## 1. Setup

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_places, available_cities
from src.recommender import TravelRecommender
from src.itinerary import generate_itinerary

print("✅ Final testing environment ready")
print("Project:", PROJECT_ROOT)


d:\college_work\PG\linkedIn_projects\TravelMate-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Final testing environment ready
Project: D:\college_work\PG\linkedIn_projects\TravelMate-AI


## 2. Load production data and engine

In [2]:
df = load_places()

embedding_path = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "multi_city_place_embeddings.npy"
)

embeddings = None

if embedding_path.exists():
    candidate = np.load(embedding_path)

    if len(candidate) == len(df):
        embeddings = candidate
        print("✅ Loaded saved multi-city embeddings")
    else:
        print(
            "⚠️ Embedding size mismatch. "
            "The engine will regenerate embeddings."
        )

engine = TravelRecommender(
    df=df,
    place_embeddings=embeddings,
)

cities = available_cities(df)

print("Dataset shape:", df.shape)
print("Cities:", cities)


✅ Loaded saved multi-city embeddings


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4558.64it/s]


Dataset shape: (120, 27)
Cities: ['Goa', 'Jaipur', 'Manali', 'Rishikesh', 'Shimla', 'Udaipur']


## 3. Verify the production recommender

In [7]:
# Verify the production recommender structure

required_base_columns = [
    "city",
    "name",
    "rating",
    "reviews",
]

missing_base = [
    col
    for col in required_base_columns
    if col not in engine.df.columns
]

print("Missing base columns:", missing_base)

assert not missing_base

print("✅ Production recommender contains required base fields")

Missing base columns: []
✅ Production recommender contains required base fields


## 4. Define final test scenarios

In [8]:
TEST_SCENARIOS = {
    "nature_photographer": {
        "query": (
            "peaceful scenic places surrounded by "
            "nature for photography"
        ),
        "preferences": {
            "nature": 1.0,
            "history": 0.1,
            "culture": 0.2,
            "adventure": 0.4,
            "photography": 1.0,
            "shopping": 0.1,
            "religious": 0.0,
            "family": 0.3,
        },
    },

    "history_culture": {
        "query": (
            "historical cultural forts palaces "
            "temples and heritage places"
        ),
        "preferences": {
            "nature": 0.1,
            "history": 1.0,
            "culture": 1.0,
            "adventure": 0.0,
            "photography": 0.8,
            "shopping": 0.3,
            "religious": 0.3,
            "family": 0.3,
        },
    },

    "adventure": {
        "query": (
            "adventure trekking outdoor and "
            "exciting travel experiences"
        ),
        "preferences": {
            "nature": 0.7,
            "history": 0.1,
            "culture": 0.1,
            "adventure": 1.0,
            "photography": 0.5,
            "shopping": 0.0,
            "religious": 0.0,
            "family": 0.2,
        },
    },

    "family": {
        "query": (
            "family friendly relaxing places "
            "and activities"
        ),
        "preferences": {
            "nature": 0.7,
            "history": 0.2,
            "culture": 0.3,
            "adventure": 0.1,
            "photography": 0.4,
            "shopping": 0.2,
            "religious": 0.1,
            "family": 1.0,
        },
    },

    "shopping_culture": {
        "query": (
            "local markets shopping and "
            "cultural experiences"
        ),
        "preferences": {
            "nature": 0.1,
            "history": 0.5,
            "culture": 0.8,
            "adventure": 0.0,
            "photography": 0.4,
            "shopping": 1.0,
            "religious": 0.2,
            "family": 0.3,
        },
    },
}

print(
    "Scenarios:",
    list(TEST_SCENARIOS.keys())
)


Scenarios: ['nature_photographer', 'history_culture', 'adventure', 'family', 'shopping_culture']


## 5. Test recommendation pipeline

In [9]:
recommendation_tests = []

for city in cities:

    for scenario_name, scenario in TEST_SCENARIOS.items():

        start = time.perf_counter()

        result = engine.recommend(
            destination=city,
            query=scenario["query"],
            user_preferences=scenario["preferences"],
            top_n=5,
        )

        elapsed_ms = (
            time.perf_counter() - start
        ) * 1000

        recommendation_tests.append({
            "city": city,
            "scenario": scenario_name,
            "rows": len(result),
            "city_correct": (
                not result.empty
                and result["city"].eq(city).all()
            ),
            "has_final_score": (
                "final_score" in result.columns
                and result["final_score"].notna().all()
            ),
            "has_coordinates": (
                result["latitude"].notna().all()
                and result["longitude"].notna().all()
            ),
            "response_time_ms": round(
                elapsed_ms,
                2
            ),
        })

recommendation_test_df = pd.DataFrame(
    recommendation_tests
)

recommendation_test_df.head(10)


,city,scenario,rows,city_correct,has_final_score,has_coordinates,response_time_ms
0,Goa,nature_photographer,5,True,True,True,52.04
1,Goa,history_culture,5,True,True,True,27.73
2,Goa,adventure,5,True,True,True,20.43
3,Goa,family,5,True,True,True,23.88
4,Goa,shopping_culture,5,True,True,True,23.20
5,Jaipur,nature_photographer,5,True,True,True,22.04
6,Jaipur,history_culture,5,True,True,True,21.54
7,Jaipur,adventure,5,True,True,True,23.28
8,Jaipur,family,5,True,True,True,17.98
9,Jaipur,shopping_culture,5,True,True,True,24.52


## 6. Recommendation hard checks

In [10]:
assert (
    recommendation_test_df["rows"] > 0
).all()

assert (
    recommendation_test_df["city_correct"]
).all()

assert (
    recommendation_test_df["has_final_score"]
).all()

assert (
    recommendation_test_df["has_coordinates"]
).all()

print(
    "✅ All recommendation scenarios passed"
)


✅ All recommendation scenarios passed


## 7. Test itinerary pipeline

In [11]:
def prepare_for_itinerary(result):
    result = result.copy()

    text = (
        result["name"].fillna("").astype(str)
        + " "
        + result["category"].fillna("").astype(str)
    ).str.lower()

    def activity(value):
        if "waterfall" in value or "falls" in value:
            return "waterfall"
        if "rafting" in value:
            return "rafting"
        if "trek" in value:
            return "trekking"
        if "viewpoint" in value or "view point" in value:
            return "viewpoint"
        if any(
            x in value
            for x in [
                "temple",
                "church",
                "mosque",
                "gurudwara",
                "monastery",
            ]
        ):
            return "religious_site"
        if any(
            x in value
            for x in [
                "fort",
                "palace",
                "museum",
                "heritage",
                "castle",
            ]
        ):
            return "heritage"
        if any(
            x in value
            for x in [
                "market",
                "bazaar",
                "mall",
                "shopping",
            ]
        ):
            return "shopping"
        if any(
            x in value
            for x in [
                "beach",
                "lake",
                "river",
                "park",
                "forest",
                "garden",
            ]
        ):
            return "nature"
        if "snow" in value or "ski" in value:
            return "winter_experience"

        return "sightseeing"

    result["activity_type"] = text.apply(
        activity
    )

    duration_map = {
        "waterfall": 90,
        "rafting": 120,
        "trekking": 150,
        "viewpoint": 45,
        "religious_site": 60,
        "heritage": 120,
        "shopping": 90,
        "nature": 90,
        "winter_experience": 90,
        "sightseeing": 60,
    }

    result["estimated_visit_minutes"] = (
        result["activity_type"]
        .map(duration_map)
        .fillna(60)
        .astype(int)
    )

    result["estimated_price_level"] = 1

    return result


## 8. Run itinerary tests across all cities

In [12]:
itinerary_tests = []

for city in cities:

    scenario = TEST_SCENARIOS[
        "nature_photographer"
    ]

    recommendations = engine.recommend(
        destination=city,
        query=scenario["query"],
        user_preferences=scenario["preferences"],
        top_n=12,
    )

    candidates = prepare_for_itinerary(
        recommendations
    )

    start = time.perf_counter()

    plan = generate_itinerary(
        candidates=candidates,
        days=3,
        max_day_minutes=7 * 60,
        average_speed_kmph=25,
    )

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000

    if plan.empty:
        budget_valid = False
        city_valid = False
        coordinate_valid = False
        days_used = 0
    else:
        totals = (
            plan
            .assign(
                total_minutes=lambda x:
                    x["travel_before_minutes"]
                    + x["visit_minutes"]
            )
            .groupby("day")["total_minutes"]
            .sum()
        )

        budget_valid = bool(
            (
                totals
                <= 7 * 60 + 1e-9
            ).all()
        )

        city_valid = bool(
            plan["city"].eq(city).all()
        )

        coordinate_valid = bool(
            plan["latitude"].notna().all()
            and plan["longitude"].notna().all()
        )

        days_used = int(
            plan["day"].nunique()
        )

    itinerary_tests.append({
        "city": city,
        "stops": len(plan),
        "days_used": days_used,
        "budget_valid": budget_valid,
        "city_valid": city_valid,
        "coordinates_valid": coordinate_valid,
        "generation_time_ms": round(
            elapsed_ms,
            2
        ),
    })

itinerary_test_df = pd.DataFrame(
    itinerary_tests
)

itinerary_test_df


,city,stops,days_used,budget_valid,city_valid,coordinates_valid,generation_time_ms
0,Goa,9,3,True,True,True,108.27
1,Jaipur,11,3,True,True,True,108.50
2,Manali,12,3,True,True,True,120.77
3,Rishikesh,12,3,True,True,True,153.70
4,Shimla,12,3,True,True,True,132.51
5,Udaipur,12,3,True,True,True,101.14


## 9. Itinerary hard checks

In [13]:
assert (
    itinerary_test_df["stops"] > 0
).all()

assert (
    itinerary_test_df["days_used"] <= 3
).all()

assert (
    itinerary_test_df["budget_valid"]
).all()

assert (
    itinerary_test_df["city_valid"]
).all()

assert (
    itinerary_test_df["coordinates_valid"]
).all()

print(
    "✅ All itinerary scenarios passed"
)


✅ All itinerary scenarios passed


## 10. Test personalization behavior

The same city should respond differently to substantially different traveller profiles.

This is a behavioral test, not a ground-truth accuracy test.


In [14]:
personalization_tests = []

city = cities[0]

results_by_profile = {}

for scenario_name, scenario in TEST_SCENARIOS.items():

    result = engine.recommend(
        destination=city,
        query=scenario["query"],
        user_preferences=scenario["preferences"],
        top_n=5,
    )

    results_by_profile[
        scenario_name
    ] = set(
        result["name"].astype(str)
    )

for i, (name_a, set_a) in enumerate(
    results_by_profile.items()
):

    for name_b, set_b in list(
        results_by_profile.items()
    )[i + 1:]:

        union = set_a | set_b

        similarity = (
            len(set_a & set_b) / len(union)
            if union
            else 1.0
        )

        personalization_tests.append({
            "profile_a": name_a,
            "profile_b": name_b,
            "jaccard_similarity":
                similarity,
            "ranking_change":
                1.0 - similarity,
        })

personalization_test_df = pd.DataFrame(
    personalization_tests
)

personalization_test_df


,profile_a,profile_b,jaccard_similarity,ranking_change
0,nature_photographer,history_culture,0.000000,1.000000
1,nature_photographer,adventure,0.428571,0.571429
2,nature_photographer,family,0.428571,0.571429
3,nature_photographer,shopping_culture,0.000000,1.000000
4,history_culture,adventure,0.000000,1.000000
5,history_culture,family,0.000000,1.000000
6,history_culture,shopping_culture,0.428571,0.571429
7,adventure,family,0.666667,0.333333
8,adventure,shopping_culture,0.000000,1.000000
9,family,shopping_culture,0.111111,0.888889


A completely identical recommendation set for every profile would be a warning sign.

The current production engine has already shown positive personalization sensitivity during Notebook 23.


In [15]:
assert not personalization_test_df.empty

print(
    "Mean personalization change:",
    round(
        personalization_test_df[
            "ranking_change"
        ].mean(),
        4
    )
)

print(
    "✅ Personalization behavior checked"
)


Mean personalization change: 0.7937
✅ Personalization behavior checked


## 11. Test invalid destination handling

In [16]:
invalid_destination_passed = False

try:
    engine.recommend(
        destination="Atlantis",
        query="beautiful places",
        user_preferences=TEST_SCENARIOS[
            "family"
        ]["preferences"],
        top_n=5,
    )

except ValueError as exc:

    invalid_destination_passed = (
        "not found" in str(exc).lower()
        or "available" in str(exc).lower()
    )

assert invalid_destination_passed

print(
    "✅ Invalid destination handled correctly"
)


✅ Invalid destination handled correctly


## 12. Test repeated calls for stability

In [17]:
stability_scenario = TEST_SCENARIOS[
    "nature_photographer"
]

outputs = []

for _ in range(3):

    result = engine.recommend(
        destination="Manali"
        if "Manali" in cities
        else cities[0],
        query=stability_scenario["query"],
        user_preferences=stability_scenario[
            "preferences"
        ],
        top_n=5,
    )

    outputs.append(
        list(
            result["name"].astype(str)
        )
    )

stable = all(
    output == outputs[0]
    for output in outputs[1:]
)

assert stable

print(
    "✅ Repeated recommendation calls are stable"
)


✅ Repeated recommendation calls are stable


## 13. HTTP-level FastAPI test

This section verifies the actual running service.

Start FastAPI first:

```powershell
uvicorn app.main:app --reload
```

Then run the cell below.


In [18]:
API_BASE_URL = "http://127.0.0.1:8000"

api_status = None
api_error = None

try:
    response = requests.get(
        f"{API_BASE_URL}/health",
        timeout=20,
    )

    api_status = response.status_code

except Exception as exc:
    api_error = str(exc)

print(
    "Status:",
    api_status
)

if api_error:
    print(
        "API error:",
        api_error
)


Status: 200


## 14. HTTP endpoint checks

In [19]:
if api_status == 200:

    health = requests.get(
        f"{API_BASE_URL}/health",
        timeout=20,
    ).json()

    cities_response = requests.get(
        f"{API_BASE_URL}/cities",
        timeout=20,
    ).json()

    assert health["status"] == "healthy"

    api_cities = set(
        cities_response["cities"]
    )

    assert api_cities == set(cities)

    payload = {
        "destination": "Goa"
        if "Goa" in cities
        else cities[0],
        "query": (
            "peaceful scenic places "
            "for photography"
        ),
        "days": 3,
        "top_n": 5,
        "preferences":
            TEST_SCENARIOS[
                "nature_photographer"
            ]["preferences"],
    }

    rec_response = requests.post(
        f"{API_BASE_URL}/recommend",
        json=payload,
        timeout=120,
    )

    assert rec_response.status_code == 200

    rec_json = rec_response.json()

    assert (
        rec_json["count"] > 0
    )

    assert all(
        item["city"]
        == payload["destination"]
        for item
        in rec_json[
            "recommendations"
        ]
    )

    itinerary_response = requests.post(
        f"{API_BASE_URL}/itinerary",
        json=payload,
        timeout=120,
    )

    assert (
        itinerary_response.status_code
        == 200
    )

    itinerary_json = (
        itinerary_response.json()
    )

    assert (
        itinerary_json[
            "scheduled_stops"
        ] > 0
    )

    print(
        "✅ HTTP /health passed"
    )

    print(
        "✅ HTTP /cities passed"
    )

    print(
        "✅ HTTP /recommend passed"
    )

    print(
        "✅ HTTP /itinerary passed"
    )

else:

    print(
        "⚠️ FastAPI is not running."
        " Start it and rerun this section."
    )


✅ HTTP /health passed
✅ HTTP /cities passed
✅ HTTP /recommend passed
✅ HTTP /itinerary passed


## 15. Check for the old Manali-only text

In [20]:
files_to_scan = [
    PROJECT_ROOT / "app" / "main.py",
    PROJECT_ROOT / "app" / "streamlit_app.py",
]

legacy_phrases = [
    "prototype currently supports manali only",
    "manali only",
    "trained on manali data only",
]

legacy_hits = []

for path in files_to_scan:

    if not path.exists():
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="ignore",
    ).lower()

    for phrase in legacy_phrases:

        if phrase in text:
            legacy_hits.append({
                "file": str(path),
                "phrase": phrase,
            })

legacy_df = pd.DataFrame(
    legacy_hits
)

legacy_df


""


In [21]:
assert legacy_df.empty

print(
    "✅ No Manali-only legacy restriction "
    "found in production app files"
)


✅ No Manali-only legacy restriction found in production app files


## 16. Generate final test report

In [22]:
final_report = pd.DataFrame([
    {
        "test": "Dataset loaded",
        "status": True,
    },
    {
        "test": "All cities represented",
        "status": set(cities)
        == {
            "Goa",
            "Jaipur",
            "Manali",
            "Rishikesh",
            "Shimla",
            "Udaipur",
        },
    },
    {
        "test": "Recommendation city correctness",
        "status": recommendation_test_df[
            "city_correct"
        ].all(),
    },
    {
        "test": "Recommendation scores valid",
        "status": recommendation_test_df[
            "has_final_score"
        ].all(),
    },
    {
        "test": "Recommendation coordinates valid",
        "status": recommendation_test_df[
            "has_coordinates"
        ].all(),
    },
    {
        "test": "Itinerary budget validity",
        "status": itinerary_test_df[
            "budget_valid"
        ].all(),
    },
    {
        "test": "Itinerary city validity",
        "status": itinerary_test_df[
            "city_valid"
        ].all(),
    },
    {
        "test": "Itinerary coordinates valid",
        "status": itinerary_test_df[
            "coordinates_valid"
        ].all(),
    },
    {
        "test": "Invalid city handled",
        "status": invalid_destination_passed,
    },
    {
        "test": "Repeated calls stable",
        "status": stable,
    },
    {
        "test": "Legacy Manali-only text absent",
        "status": legacy_df.empty,
    },
])

final_report


,test,status
0,Dataset loaded,True
1,All cities represented,True
2,Recommendation city correctness,True
3,Recommendation scores valid,True
4,Recommendation coordinates valid,True
5,Itinerary budget validity,True
6,Itinerary city validity,True
7,Itinerary coordinates valid,True
8,Invalid city handled,True
9,Repeated calls stable,True


## 17. Final result

In [23]:
all_passed = bool(
    final_report["status"].all()
)

print("=" * 60)
print("🌍 TRAVELMATE AI FINAL SYSTEM TEST")
print("=" * 60)

for _, row in final_report.iterrows():

    symbol = "✅" if row["status"] else "❌"

    print(
        f"{symbol} {row['test']}"
    )

print()

if all_passed:
    print(
        "🎉 ALL AUTOMATED SYSTEM TESTS PASSED"
    )
else:
    print(
        "⚠️ SOME TESTS FAILED - FIX BEFORE DEPLOYMENT"
    )

print(
    "\nFinal status:",
    "READY FOR DEPLOYMENT"
    if all_passed
    else "NOT READY"
)


🌍 TRAVELMATE AI FINAL SYSTEM TEST
✅ Dataset loaded
✅ All cities represented
✅ Recommendation city correctness
✅ Recommendation scores valid
✅ Recommendation coordinates valid
✅ Itinerary budget validity
✅ Itinerary city validity
✅ Itinerary coordinates valid
✅ Invalid city handled
✅ Repeated calls stable
✅ Legacy Manali-only text absent

🎉 ALL AUTOMATED SYSTEM TESTS PASSED

Final status: READY FOR DEPLOYMENT


## 18. Save the final testing artifacts

In [24]:
test_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "evaluation"
)

test_dir.mkdir(
    parents=True,
    exist_ok=True
)

recommendation_test_df.to_csv(
    test_dir
    / "final_recommendation_system_tests.csv",
    index=False
)

itinerary_test_df.to_csv(
    test_dir
    / "final_itinerary_system_tests.csv",
    index=False
)

personalization_test_df.to_csv(
    test_dir
    / "final_personalization_tests.csv",
    index=False
)

final_report.to_csv(
    test_dir
    / "final_system_test_report.csv",
    index=False
)

print(
    f"✅ Final test artifacts saved to: {test_dir}"
)


✅ Final test artifacts saved to: D:\college_work\PG\linkedIn_projects\TravelMate-AI\data\processed\evaluation


# 🏁 Final testing milestone

TravelMate AI has now been tested at the system level.

The final gate is:

```text
✅ All automated tests pass
        ↓
🚀 Deployment preparation
```

Before deployment, keep FastAPI and Streamlit testing in separate terminals and verify the browser once more with at least two different cities.

The next stage is packaging the project professionally:

```text
README
Architecture diagram
requirements.txt
.env.example
GitHub cleanup
Deployment configuration
Screenshots
LinkedIn project presentation
```

> 🧠 **Final engineering rule:** Don't deploy the demo because it looks good. Deploy it because the system tests say it is ready.
